# Flexible paper figures tool

Per-figure block selection, params overrides, inline build, and a dated export —
the flexible counterpart to `analysis_figure_suite.ipynb` (which stays as the
all-at-once driver).

**Workflow**
1. (Optional) Browse & stage blocks → write a paper registry YAML (section 0.5)
2. Point at a registry + params YAML and a run tag
3. Build (or load-cached) event tables once — set `USE_FINALIZED_SACCADES=True` to prefer
   per-block `analysis/saccades/` CSVs from the preprocessing GUI (see also
   `compile_block_saccades.ipynb`)
4. For each figure: tick blocks → filter saccades (concurrent/monocular, head movement, query) → tweak params → **Build**
5. Finalize into `outputs/paper_figures_<tag>_<date>_<HH>_<MM>/`

**Mouse / pogona supplementary:** re-run this notebook against
`configs/mouse_M_002_blocks.yaml` + `configs/analysis_params_mouse.yaml`.
Blocks without behavior-state files correctly show as ineligible for Fig 3c/3e/3f.

Requires `ipywidgets`, `PYTHONPATH` including `src` (set below), and the
`eye_repo_mac` (or equivalent) env.


## 0. Setup


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import FileLink, Markdown, display

%matplotlib inline

REPO = Path.cwd()
if (REPO / "src" / "eye_tracking_system_tools").is_dir():
    pass
elif (REPO.parent / "src" / "eye_tracking_system_tools").is_dir():
    REPO = REPO.parent
else:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "eye_tracking_system_tools").is_dir():
            REPO = p
            break

sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("MPLCONFIGDIR", str(REPO / ".mplconfig"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(exist_ok=True)
CWD = Path.cwd().resolve()

from eye_tracking_system_tools.analysis.block_registry import (
    load_registry,
    registry_summary,
    write_paper_registry,
)
from eye_tracking_system_tools.analysis.event_cache import build_or_load_event_tables
from eye_tracking_system_tools.analysis.export_meta import load_params_yaml
from eye_tracking_system_tools.analysis.figure_catalog import CATALOG
from eye_tracking_system_tools.analysis.jitter_gui import JitterBlockBrowser
from eye_tracking_system_tools.analysis.paper_export import (
    describe_export,
    finalize_paper_export,
    rebuild_from_export,
)
from eye_tracking_system_tools.analysis.paper_gui import (
    PaperContext,
    PaperFigureSelector,
    block_qc_table,
    make_fig1e_selector,
    make_vignette_selector,
)
from eye_tracking_system_tools.analysis.run_layout import resolve_run_dir

PAPER_REGISTRY = REPO / "configs" / "paper_blocks_custom.yaml"

def link_path(p: Path):
    p = Path(p).resolve()
    try:
        rel = os.path.relpath(p, CWD)
        display(FileLink(rel))
    except Exception:
        display(Markdown(f"[{p.name}]({p.as_uri()})"))

print("REPO:", REPO)
print("Catalog figures:", ", ".join(CATALOG))


## 0.5 Browse & build a paper registry

Use the same filesystem browser as the jitter pipeline to multi-select block
folders, then **Save registry**. With `registry_format="paper"` this writes an
`animals: {name: [paths]}` YAML (what section 1 / `load_registry` expects) —
not the jitter `blocks:` / `mount_type` schema.

Mount tags in the UI are ignored for paper registries; leave them at the default.
Prefer **Add selected** on `block_*` folders (the Scan button looks for jitter
reports, which paper figures do not need).

Skip this section if you already have a registry (e.g. `paper_blocks.yaml`).


In [ ]:
# Destination for the paper-format registry written by Save.
# Change the name if you want a dated / tagged file instead of overwriting.
PAPER_REGISTRY = REPO / "configs" / "paper_blocks_custom.yaml"

browser = JitterBlockBrowser(
    PAPER_REGISTRY,
    repo=REPO,
    registry_format="paper",  # animals: {…} — not the jitter blocks: schema
    load_existing=True,       # re-open previous custom registry if present
)
browser


In [ ]:
# After clicking Save registry above, confirm what was written.
from collections import defaultdict

print(f"staged: {len(browser.specs)} block(s)")
if browser.specs:
    # Re-save in case the user staged more after the last Save click.
    write_paper_registry(PAPER_REGISTRY, browser.specs)
    print(f"wrote: {PAPER_REGISTRY}")
    link_path(PAPER_REGISTRY)
    by: dict[str, list[str]] = defaultdict(list)
    for s in browser.specs:
        by[s.animal].append(s.block_path.name)
    for animal, names in sorted(by.items()):
        print(f"  {animal}: {', '.join(names)}")
else:
    print("Nothing staged — section 1 can still point at an existing registry.")


## 1. Registry, params, run tag

Edit the paths below. If you built a registry in section 0.5, keep
`REGISTRY = PAPER_REGISTRY`; otherwise point at `paper_blocks.yaml`,
`paper_blocks_dryrun_PV_143.yaml`, `sample_blocks.yaml`, etc.

Empty `TAG` writes to `paper_latest` (overwritable); set a tag to keep a
snapshot under `outputs/paper_<tag>/` for scratch builds. The dated finalize
folder is separate (section 4).


In [ ]:
# --- edit me ---
# Prefer the custom registry from section 0.5 when it exists; else the paper cohort.
REGISTRY = (
    PAPER_REGISTRY
    if PAPER_REGISTRY.is_file()
    else REPO / "configs" / "paper_blocks.yaml"
)
# REGISTRY = REPO / "configs" / "paper_blocks_dryrun_PV_143.yaml"
# REGISTRY = REPO / "configs" / "sample_blocks.yaml"
# REGISTRY = REPO / "configs" / "mouse_M_002_blocks.yaml"
PARAMS = REPO / "configs" / "analysis_params.yaml"
# PARAMS = REPO / "configs" / "analysis_params_mouse.yaml"
TAG = ""  # empty → paper_latest
FORCE_REBUILD_EVENTS = False  # True to ignore the event-table cache
KEEP_TRACES = False  # cache stores events only; Fig 2c/2d, 2f, 3* reload traces on Build
# Prefer preprocessing-GUI finalized CSVs under each block's analysis/saccades/
# (falls back to on-the-fly detection when missing). See compile_block_saccades.ipynb.
USE_FINALIZED_SACCADES = True
# ---------------

print("REGISTRY:", REGISTRY)

params = load_params_yaml(PARAMS)
specs = load_registry(REGISTRY)
print(registry_summary(specs))

run = resolve_run_dir(REPO / "outputs", TAG, prefix="paper", default_name="paper_latest")
print("run_dir:", run.run_dir)
print("figures:", run.figures_dir)
print("metadata:", run.metadata_dir)


## 2. Build event tables (cached) + QC

Detection is the expensive step. Results are cached under
`metadata/event_cache/<sha1>.pkl` keyed on block paths + saccade/binocular params.
A restarted kernel reloads instantly; Fig 2c/2d, 2f, and 3* reload traces for
selected blocks only (kernel 2c/2d needs `k_phi`/`k_theta`/`ms_axis`).


In [ ]:
tables, cache_path, from_cache = build_or_load_event_tables(
    specs,
    params,
    run.metadata_dir,
    keep_traces=KEEP_TRACES,
    force=FORCE_REBUILD_EVENTS,
    prefer_finalized=USE_FINALIZED_SACCADES,
)
print(("loaded from cache" if from_cache else "built fresh"), "→", cache_path)
print(
    f"blocks={len(tables.blocks)}  all_saccades={len(tables.all_saccades)}  "
    f"synced_rows={len(tables.synced)}  non_synced={len(tables.non_synced)}"
)

ctx = PaperContext(
    tables,
    run.run_dir,
    registry_path=REGISTRY,
    params_path=PARAMS,
)
qc = block_qc_table(tables)
display(qc)
print(
    "behavior_state:", int(qc["has_behavior_state"].sum()), "/", len(qc),
    " | pix_size:", int(qc["has_pix_size"].sum()), "/", len(qc),
)


## 3. Per-figure selectors

Each cell is independent: tick blocks, optionally filter saccades, edit the YAML
params box, click **Build**.

**Saccade filter** (on every event-based figure):
- **Events** — All / Concurrent (synced pairs) / Monocular
- **Head** — Any / Without head / With head / Labeled only
- **Advanced** — pandas `query` string and per-column truth filters for other
  boolean-like flags

The live count under the filter shows how many events remain after the current
block selection + filter. Filters are stored in the export bundle and replayed
by `rebuild_from_export`.


### Fig 2c / 2d — amp-binned position & velocity


In [ ]:
sel_2c_2d = PaperFigureSelector('2c_2d', ctx)
sel_2c_2d


### Fig 2e — amplitude–velocity linear fit


In [ ]:
sel_2e = PaperFigureSelector('2e', ctx)
sel_2e


### Fig 2f — inter-ocular peak-speed coupling (needs traces)


In [ ]:
sel_2f = PaperFigureSelector('2f', ctx)
sel_2f


### Fig 2g — amplitude distributions


In [ ]:
sel_2g = PaperFigureSelector('2g', ctx)
sel_2g


### Fig 2h — endpoint heatmaps


In [ ]:
sel_2h = PaperFigureSelector('2h', ctx)
sel_2h


### Fig 2i — polar direction histograms


In [ ]:
sel_2i = PaperFigureSelector('2i', ctx)
sel_2i


### Fig 2j — orientation tuning


In [ ]:
sel_2j = PaperFigureSelector('2j', ctx)
sel_2j


### Fig 3d — inter-saccade interval densities


In [ ]:
sel_3d = PaperFigureSelector('3d', ctx)
sel_3d


### Fig 3e — pupil by behavioral state


In [ ]:
sel_3e = PaperFigureSelector('3e', ctx)
sel_3e


### Fig 3f — z-scored pupil state difference


In [ ]:
sel_3f = PaperFigureSelector('3f', ctx)
sel_3f


### Fig 3a — quiet vignette (single block + time window)


In [ ]:
sel_3a = make_vignette_selector("3a", ctx, start_s=210.0, end_s=240.0)
sel_3a


### Fig 3b — active vignette


In [ ]:
sel_3b = make_vignette_selector("3b", ctx, start_s=310.0, end_s=340.0)
sel_3b


### Fig 3c — full vignette with state / rates


In [ ]:
sel_3c = make_vignette_selector("3c", ctx, start_s=200.0, end_s=415.0)
sel_3c


### Fig 1e — camera jitter (from a finalized jitter export)

Point at a folder produced by `jitter_mount_pipeline.ipynb` section 6
(`jitter_comparison_figures_<tag>_<date>_…`), or its `jitter_comparison_data.pickle`.


In [ ]:
# Optional default: newest jitter export under outputs/, else leave blank
_jitter_candidates = sorted((REPO / "outputs").glob("jitter_comparison_figures_*"), reverse=True)
JITTER_BUNDLE = _jitter_candidates[0] if _jitter_candidates else None
print("default jitter bundle:", JITTER_BUNDLE)

sel_1e = make_fig1e_selector(ctx, JITTER_BUNDLE)
sel_1e


## 4. Finalize export

Creates `outputs/paper_figures_<tag>_<YYYYmmdd>_<HH>_<MM>/` with the PDFs that
were built, plus `figure_specs.pickle`, `selections.csv`, and `manifest.yaml`.
Only selections + params + registry pointers are stored — rebuild re-runs from
source blocks.


In [ ]:
EXPORT_TAG = TAG or "dryrun"  # edit
EXPORT_NOTES = ""

if not ctx.builds:
    raise RuntimeError("Build at least one figure in section 3 before finalizing.")

result = finalize_paper_export(
    ctx.builds,
    REPO / "outputs",
    tag=EXPORT_TAG,
    registry_path=REGISTRY,
    params_path=PARAMS,
    notes=EXPORT_NOTES,
    scratch_figures_dir=run.figures_dir,
)
print(result)
link_path(result.export_dir)
link_path(result.manifest_path)
display(describe_export(result.export_dir))


## 5. Rebuild from export (demo)

Re-runs each recorded figure from its stored selection and params. Needs the
same data volumes mounted and event tables covering those blocks.


In [ ]:
# Point at the export you just made (or any earlier one)
EXPORT_PATH = result.export_dir  # or Path(".../paper_figures_...")

rebuild_dir = run.run_dir / "rebuild_demo"
rebuild_dir.mkdir(parents=True, exist_ok=True)
rebuilt = rebuild_from_export(EXPORT_PATH, tables, rebuild_dir, show=False)
print(f"rebuilt {len(rebuilt)} figure(s) → {rebuild_dir}")
for fig_id, paths in rebuilt.items():
    print(fig_id, ":", ", ".join(Path(p).name for p in paths.values() if str(p).endswith('.pdf')))


## Notes

| Topic | Detail |
|---|---|
| Suite notebook | `analysis_figure_suite.ipynb` remains the all-at-once driver |
| Scratch vs finalize | Scratch: `outputs/paper_<tag|latest>/`; finalize: dated `paper_figures_*` |
| Event cache | `metadata/event_cache/<sha1>.pkl` — events only, no traces |
| Fig 1e | Replots a finalized jitter export (µm); single source of truth with the jitter tool |
| Mouse registry | `configs/mouse_M_002_blocks.yaml` + `analysis_params_mouse.yaml` |
| FileLink cwd | Kernel cwd is `development/`; helpers above use `os.path.relpath` |
